In [1]:
import requests
import pandas as pd
import time
import json


In [2]:
def OpenAlexAPI(url, params):
    
    # Initialize cursor
    cursor = "*"

    # Create a container
    all_datasets = []
    page = 0
    while cursor:
        params["cursor"] = cursor
        
        # Check for request success
        response = requests.get(url, params=params)
        if response.status_code == 200:
            print(f"Status code: {response.status_code}", ';',  "Page number: ", page)
        if response.status_code != 200:
            print("Oh no! Something went wrong during the live demo! How embarrassing!")
            response.raise_for_status()
        records = response.json()['results']
        
        # create a loop iterator
        record_number = 0
        page += 1
       
       # Loop over all the records
        for record in records:
            record_number += 1
            #print("Page number: ", page, 'Record number:', record_number, record)
            all_datasets.append({'id': record['id'],
                                'doi': record['doi'],
                                'title': record['title'],
                                'type': record['type'],
                                'publication_year': record['publication_year'],
                                'publication_date': record['publication_date'],
                                'topics': record['topics'][:],
                                'language': record['language'],
                                'retracted': record['is_retracted'],
                                'index location': record['indexed_in'],
                                'primary_location': record['primary_location']['source'],
                                'is_oa': record['open_access']['is_oa'],
                                'oa_status': record['open_access']['oa_status'],
                                'license': record['primary_location']['license'], 
                                'authors': record['authorships'][:],
                                'affiliations': record['authorships'][0]['affiliations'],
                                'Institution': record['authorships'][0]['institutions'],
                                'apc_list': record['apc_list'],
                                'apc_paid': record['apc_paid'],
                                'fwci': record['fwci'],
                                'has_fulltext': record['has_fulltext'],
                                'cited_by_count': record['cited_by_count'], 
                                'citation_normalized_percentile': record['citation_normalized_percentile'],
                                'cited_by_percentile_year': record['cited_by_percentile_year'],
                                'biblio': record['biblio']
                                })         

        # Update cursor
        cursor = response.json()['meta']['next_cursor']
        
        time.sleep(3)
    return all_datasets

parameters = {
    "per-page": 100,
}   
# types/dataset,
#parameters = {
    #"filter": f"authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549",  # University of Tasmania
    #"per-page": 100,
    #"select": "id,doi,publication_year,title,primary_location,authorships,topics",
#}


In [3]:
# API Call to scrape UMN Data
#with open('OpenAlexDataCitation.json', 'w') as outfile:
#    json.dump(OpenAlexAPI('https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i130238516', parameters), outfile)
  
#df = pd.read_json('OpenAlexDataCitation.json')
#df

In [ ]:
with open('OpenAlexDataCitation.json', 'w') as outfile:
    json.dump(OpenAlexAPI('https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i57206974|i130238516|i2801744472|i27837315|i157725225|i2802508227|i4210135927|i188538660|i205783295|i859038795|i135310074|i20089843|i130769515|i51556381|i79576946|i170897317|i204465549|i169521973|i114395901', parameters), outfile)

Oh no! Something went wrong during the live demo! How embarrassing!


HTTPError: 403 Client Error: FORBIDDEN for url: https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i57206974%20%20%20%20%20%20%20%20%20&per-page=100&cursor=%2A

In [ ]:
# Function that creates new dataframe from nested list
def CleanJSON(column,InputDF):
    df = pd.concat([pd.DataFrame(x) for x in InputDF[column]], keys=InputDF.index).reset_index(level=1,drop=True)
    return df

In [ ]:
def missing_index(df):
    missing_indices = ([i for i in range(0, len(df)) if i not in df.index])
    return missing_indices

In [ ]:
pd.set_option("display.max_columns", 55)


In [ ]:
df = pd.read_json('OpenAlexDataCitation.json')
df.head()

In [ ]:
# investigate the columns
df.columns

In [ ]:
# Remove list element from column
df['index location'] = df['index location'].str[0]
df

In [ ]:
# normalize  citation metric columns data

citation_normalized_percentile_df = pd.json_normalize(df['citation_normalized_percentile'])
cited_by_percentile_year_df = pd.json_normalize(df['cited_by_percentile_year'])
biblio_df = pd.json_normalize(df['biblio'])
# concat normalized data into original df

df = pd.concat([df, citation_normalized_percentile_df], axis=1)
df = pd.concat([df, cited_by_percentile_year_df], axis=1)
df = pd.concat([df, biblio_df], axis=1)

#r ename column
df = df.rename(columns={'value': 'citation_normalized_value'})


df.head()

In [ ]:
# normalize columns data
primary_location_df = pd.json_normalize(df['primary_location'])


# extract only display name column
primary_location_df = primary_location_df[['display_name']]

#rename display name column
primary_location_df = primary_location_df.rename(columns={'display_name': 'Repository'})

#primary_location_df = primary_location_df.reindex(range(primary_location_df.index.min(), primary_location_df .index.max()+1))

# concat display name column to main df
df = pd.concat([df, primary_location_df], axis=1)

In [ ]:
# Use function to normalize Authors nested list
authors_df = CleanJSON('authors', df)
authors_df['raw_affiliation_strings'] = authors_df['raw_affiliation_strings'].str[0]
authors_df

In [ ]:
missing_index(authors_df['raw_affiliation_strings'])

In [ ]:
df.columns

In [ ]:
# Create new DataFrame to concat original data from
cleaned_df = df[['id', 'doi', 'type', 'title', 'publication_year', 'publication_date', 'topics',
       'language', 'retracted', 'index location', 'is_oa',
       'oa_status', 'license', 'Repository',
       'apc_list', 'apc_paid', 'fwci', 'has_fulltext',
       'institution_assertions', 'cited_by_count',
       'citation_normalized_value', 'is_in_top_1_percent',
       'is_in_top_10_percent', 'min', 'max', 'volume', 'issue', 'first_page',
       'last_page',]].copy()
cleaned_df

In [ ]:
# concat original data to unnested cleaned_df
cleaned_df = pd.concat([authors_df, cleaned_df], axis=1)
cleaned_df


In [ ]:
# reset index to concat with author information_df
cleaned_df = cleaned_df.reset_index()

# normalize author column
author_info_df= pd.json_normalize(cleaned_df['author'])
author_info_df = author_info_df.rename(columns={'id': 'author_id', 'display_name': 'author_name'})
author_info_df

In [ ]:
# concat author_info_df to cleaned_df
cleaned_df = pd.concat([cleaned_df, author_info_df], axis=1).reset_index(drop=True)
cleaned_df.head()

In [ ]:
# normalize nested Insituttion column
institution_df = CleanJSON('institutions', cleaned_df)

# remove list format for lineage
institution_df['lineage'] = institution_df['lineage'].str[0]
institution_df

# rename columns
institution_df = institution_df.rename(columns={'id': 'institution_id', 'display_name': 'institution_name', 'type': 'funding_type'})

In [ ]:
# remove duplicate indices 
institution_df = institution_df[~institution_df.index.duplicated(keep='first')]

# fill nan values on missinging indices
institution_df = institution_df.reindex(range(institution_df.index.min(), institution_df.index.max()+1))

In [ ]:
institution_df.head(25)

In [ ]:
missing_index(institution_df)

In [ ]:
#  Use afiliations column that is already fixed
# normalize nested  column
#affiliations_df = CleanJSON('affiliations', authors_df)

# remove list format for lineage
#affiliations_df['institution_ids'] = affiliations_df['institution_ids'].str[0]

# rename columns
#affiliations_df = affiliations_df.rename(columns={'institution_ids': 'affiliations_lineage'})
#affiliations_df

#affiliations_df

In [ ]:
#missing_index(affiliations_df)

In [ ]:
#len(affiliations_df)

In [ ]:
#remove duplicate indices 
#affiliations_df = affiliations_df.reset_index()
#affiliations_df = affiliations_df.reindex(range(affiliations_df.index.min(), affiliations_df.index.max()+1))
#affiliations_df

In [ ]:
#  check if there are any mission indices
#missing_index(affiliations_df)

In [ ]:
topics_df = CleanJSON('topics', cleaned_df)
topics_df = topics_df.rename(columns={'id': 'topic_id', 'display_name': 'topic_name'})
topics_df


In [ ]:
topics_df = topics_df[~topics_df.index.duplicated(keep='first')]

topics_df = topics_df.reindex(range(topics_df.index.min(), topics_df.index.max()+1))


In [ ]:
topics_df

In [ ]:
missing_index(topics_df)

In [ ]:
subfield_df = pd.json_normalize(topics_df['subfield'])
field_df = pd.json_normalize(topics_df['field'])
domain_df = pd.json_normalize(topics_df['domain'])

# rename 
subfield_df = subfield_df.rename(columns={'id': 'subfield_id', 'display_name': 'subfield_name'})
field_df = field_df.rename(columns={'id': 'field_id', 'display_name': 'field_name'})
domain_df = domain_df.rename(columns={'id': 'domain_id', 'display_name': 'domain_name'})

field_df

In [ ]:
topics_df = topics_df[['topic_id', 'topic_name']]

In [ ]:
topics_df = pd.concat([topics_df, subfield_df], axis=1).reset_index(drop=True)
topics_df = pd.concat([topics_df, field_df], axis=1).reset_index(drop=True)
topics_df = pd.concat([topics_df, domain_df], axis=1).reset_index(drop=True)
topics_df

In [ ]:
# concat insitution_df and topics_df
#institution_df = institution_df.reset_index()
#merged_list = [institution_df, affiliations_df, topics_df]

merged_list = [institution_df, topics_df]
merged_df = pd.concat(merged_list, axis=1)
merged_df


In [ ]:
unnested_df = cleaned_df.copy()
unnested_df

In [ ]:
unnested_df.columns

In [ ]:
unnested_df = unnested_df[['author_position', 'countries',
       'is_corresponding', 'raw_author_name', 'raw_affiliation_strings',
       'id', 'doi', 'type', 'title', 'publication_year',
       'publication_date', 'language', 'apc_list', 'apc_paid',
       'fwci', 'has_fulltext', 'institution_assertions', 'cited_by_count',
       'citation_normalized_value', 'is_in_top_1_percent',
       'is_in_top_10_percent', 'min', 'max', 'volume', 'issue', 'first_page',
       'last_page', 'retracted', 'index location',
       'is_oa', 'oa_status', 'license', 'Repository', 'author_id',
       'author_name', 'orcid']]
unnested_df

In [ ]:
unnested_df = pd.concat([unnested_df, merged_df], axis=1)
unnested_df


In [ ]:
# check for nan authors
unnested_df['author_name'].isnull().values.any()

In [ ]:
# remove rows where authors are NaN
#unnested_df = unnested_df.iloc[:10568]
#unnested_df

In [ ]:
unnested_df.columns

In [ ]:
unnested_df = unnested_df[['id', 'doi', 'type', 'funding_type', 'title', 'publication_year', 'publication_date', 'language', 
       'author_id','author_name', 'author_position', 'orcid',
       'apc_list', 'apc_paid', 'fwci', 'has_fulltext', 'institution_assertions', 'cited_by_count', 'citation_normalized_value', 
              'is_in_top_1_percent', 'is_in_top_10_percent', 'min', 'max', 'volume', 'issue', 'first_page', 'last_page', 
       'topic_id', 'topic_name', 'subfield_id', 'subfield_name', 'field_id', 'field_name', 'domain_id', 'domain_name', 
       'retracted', 'index location', 'Repository', 'is_oa', 'oa_status', 'license',  
       'institution_id', 'institution_name', 'raw_affiliation_strings',  'country_code', 'ror', 'lineage']]
unnested_df

In [ ]:
# subset to ror ids
ror_df = unnested_df[(unnested_df['ror'] == 'https://ror.org/0190ak572') | (unnested_df['ror'] == 'https://ror.org/017zqws13') | (unnested_df['ror'] == 'https://ror.org/00jmfr291') 
                                            | (unnested_df['ror'] == 'https://ror.org/00x6h5n95') | (unnested_df['ror'] == 'https://ror.org/00jmfr291') | (unnested_df['ror'] == 'https://ror.org/047426m28') 
                                            | (unnested_df['ror'] == 'https://ror.org/03arq3225') | (unnested_df['ror'] == 'https://ror.org/04sac7215') | (unnested_df['ror'] == 'https://ror.org/02ttsq026') 
                                            | (unnested_df['ror'] == 'https://ror.org/05bnh6r87') | (unnested_df['ror'] == 'https://ror.org/02smfhw86') | (unnested_df['ror'] == 'https://ror.org/01y2jtd41') 
                                            | (unnested_df['ror'] == 'https://ror.org/00hx57361') | (unnested_df['ror'] == 'https://ror.org/04p491231') | (unnested_df['ror'] == 'https://ror.org/0153tk833')
                                            | (unnested_df['ror'] == 'https://ror.org/00b30xv10') | (unnested_df['ror'] == 'https://ror.org/00py81415') | (unnested_df['ror'] == 'https://ror.org/01yc7t268') 
                                            | (unnested_df['ror'] == 'https://ror.org/05fs6jp91') | (unnested_df['ror'] == 'https://ror.org/043mer456')].reset_index(drop=True)
ror_df

In [ ]:
# Only authors from DCN institutions
ror_df.to_csv('OpenAlexDataCitation.csv')